# masks

> Clean up and encode segmentation masks. Works for binary masks and for multi-category (integer-labelled) masks.

Hand-drawn or weakly-labelled masks are often ragged. Two jobs live here: (1) snap connected components to clean shapes, and (2) convert between the integer-label representation used on disk and the one-hot representation used by losses.

In [ ]:
#| default_exp masks

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import json
from pathlib import Path
from typing import Optional, Sequence

import cv2
import numpy as np

# cv2.drawContours silently no-ops on 64-bit Linux when passed int64 arrays.
# np.intp == int64 on 64-bit systems; np.int32 is safe on Linux AND Windows.
# Use this constant everywhere we pass contour coordinate arrays to OpenCV.
_CV_CONTOUR_DTYPE = np.int32

## A synthetic mask

We draw a rotated rectangle and a noisy blob so we can see the cleanup work. On real data, load with `cv2.imread(path, cv2.IMREAD_GRAYSCALE)`.

In [ ]:
raw = np.zeros((80, 80), np.uint8)
box = cv2.boxPoints(((30, 30), (24, 10), 25.0)).astype(np.int32)
cv2.fillPoly(raw, [box], 255)
rng = np.random.default_rng(0)
raw[55:65, 55:65] = (rng.random((10, 10)) > 0.4).astype(np.uint8) * 255  # ragged blob
print('unique values:', np.unique(raw))

## Binarise

Annotators save masks as `0/1` or `0/255`. `binarize` collapses either into a clean `{0, 1}` uint8.

In [ ]:
# experiment
b = (raw > 127).astype(np.uint8)
print('after binarise, unique:', np.unique(b))

In [ ]:
#| export
def binarize(
    mask: np.ndarray,    # mask with any positive encoding (1 or 255)
    thresh: int = 127,   # threshold when values are 0/255; ignored for 0/1 masks
) -> np.ndarray:
    """Collapse a mask to a `{0, 1}` uint8 array.

    Handles both `{0, 1}` and `{0, 255}` encodings automatically.

    Example:
        >>> binarize(np.array([[0, 255], [255, 0]], np.uint8)).tolist()
        [[0, 1], [1, 0]]
        >>> binarize(np.array([[0, 1], [1, 0]], np.uint8)).tolist()
        [[0, 1], [1, 0]]
    """
    if mask.max() <= 1:
        return (mask > 0).astype(np.uint8)
    return (mask > thresh).astype(np.uint8)

In [ ]:
# test
assert set(np.unique(binarize(raw)).tolist()) <= {0, 1}
assert binarize(np.array([[0, 1], [1, 0]], np.uint8)).tolist() == [[0, 1], [1, 0]]

## Snap blobs to rotated rectangles

Replace every connected component with the minimum-area rotated rectangle that encloses it. This removes ragged pixels and is the right prior when the targets are physically rectangular (pins, pads, dies).

In [ ]:
#| export
def fit_rotated_rects_mask(
    mask: np.ndarray,    # binary {0,1} (or 0/255) mask, any shape
    min_pts: int = 5,    # contours with fewer points are kept as-is
    min_area: int = 4,   # drop specks smaller than this (noise)
) -> np.ndarray:
    """Replace each connected blob with its min-area rotated bounding rectangle.

    Example:
        >>> clean = fit_rotated_rects_mask(np.eye(8, dtype=np.uint8))
        >>> clean.dtype, int(clean.max())
        (dtype('uint8'), 1)
    """
    m = binarize(mask)
    contours, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    out = np.zeros_like(m, dtype=np.uint8)
    for cnt in contours:
        if cv2.contourArea(cnt) < min_area:
            continue
        if len(cnt) < min_pts:
            cv2.drawContours(out, [cnt], -1, 1, thickness=cv2.FILLED)
            continue
        box = cv2.boxPoints(cv2.minAreaRect(cnt)).astype(_CV_CONTOUR_DTYPE)
        cv2.drawContours(out, [box], 0, 1, thickness=cv2.FILLED)
    return out

def fit_rotated_rects_multiclass(
    mask: np.ndarray,    # integer-labelled mask (0 = background, 1..K = classes)
    num_classes: int,    # number of foreground classes K
    min_pts: int = 5,    # see fit_rotated_rects_mask
    min_area: int = 4,   # see fit_rotated_rects_mask
) -> np.ndarray:
    """Apply `fit_rotated_rects_mask` independently to each foreground label.

    Later labels win where rectangles overlap.
    """
    out = np.zeros_like(mask, dtype=mask.dtype)
    for label in range(1, num_classes + 1):
        cleaned = fit_rotated_rects_mask((mask == label).astype(np.uint8), min_pts, min_area)
        out[cleaned > 0] = label
    return out

In [ ]:
# experiment + test: cleanup fills the rotated rectangle and removes ragged noise
clean = fit_rotated_rects_mask(raw)
print('raw foreground px :', int(binarize(raw).sum()))
print('clean foreground px:', int(clean.sum()))
assert clean.dtype == np.uint8 and int(clean.max()) <= 1
# multiclass: two labels stay separate
mc = np.zeros((40, 40), np.uint8); mc[5:15, 5:15] = 1; mc[25:35, 25:35] = 2
mcc = fit_rotated_rects_multiclass(mc, num_classes=2)
assert set(np.unique(mcc).tolist()) == {0, 1, 2}

## One-hot encoding

Losses operate on per-class channels. `one_hot` turns an `(H, W)` integer-label mask into a `(num_classes, H, W)` float tensor-friendly array. For binary tasks (`num_classes == 1`) it returns the single foreground channel.

In [ ]:
#| export
def one_hot(
    mask: np.ndarray,   # (H, W) integer labels: 0 = background, 1..K = classes
    num_classes: int,   # K, the number of foreground classes
) -> np.ndarray:
    """Encode an integer-label mask as `(num_classes, H, W)` float32.

    Binary (`num_classes == 1`) returns the foreground channel only.
    Multi-class returns one channel per foreground label (background dropped).

    Example:
        >>> m = np.array([[0, 1], [2, 0]]); one_hot(m, 2).shape
        (2, 2, 2)
    """
    if num_classes == 1:
        return binarize(mask)[None].astype(np.float32)
    return np.stack([(mask == c) for c in range(1, num_classes + 1)], axis=0).astype(np.float32)

In [ ]:
# test
m = np.array([[0, 1], [2, 0]])
oh = one_hot(m, 2)
assert oh.shape == (2, 2, 2)
assert np.array_equal(oh.argmax(0) + 1, np.where(m > 0, m, 1))  # foreground labels recovered
assert one_hot(binarize(raw), 1).shape == (1, 80, 80)

## Masks to COCO instances (and back)

Instance detectors such as RF-DETR segmentation (see `docs/superpowers/specs/2026-11-22-rfdetr-seg-integration-research.md`) train from a COCO folder, not from per-pixel masks. This section is the bridge, and it runs fully on plain numpy: no `rfdetr`, no `pycocotools`.

The plan, one step at a time:

1. Split a binary mask into **instances** with connected components (8-connectivity).
2. Turn each instance into **polygons**. Contours are traced on a 2x upsampled copy so that a 1-px line or a 2-px dot keeps a real, non-zero area (a contour through pixel centres has zero area for those, and a COCO loader would rasterise it to nothing).
3. Polygons cannot hold **holes** (COCO loaders union the polygon list). A component with a hole is therefore written as **RLE** (run-length encoding) instead; `segmentation='auto'` does this, `'polygon'` forces polygons and fills holes, `'rle'` always uses RLE.
4. Write the **inverse** (`coco_instances_to_mask`) so we can measure what the round trip loses.
5. Write the **COCO folder** from a patch cache.

### A synthetic test set: thin lines, holes, 2-px dots

In [ ]:
# experiment: the cases the round trip must survive
cases = {}
m = np.zeros((64, 64), np.uint8); cv2.line(m, (3, 5), (60, 40), 1, 1); cases['thin_diagonal_line'] = m
m = np.zeros((64, 64), np.uint8); m[10, 5:60] = 1; cases['thin_horizontal_line'] = m
m = np.zeros((64, 64), np.uint8); m[10:40, 10:40] = 1; m[20:30, 20:30] = 0; cases['square_with_hole'] = m
m = np.zeros((64, 64), np.uint8); m[5:7, 5:7] = 1; m[30:32, 40:42] = 1; m[50, 50] = 1; cases['small_dots'] = m
m = np.zeros((64, 64), np.uint8); m[8:30, 8:30] = 1; m[16:22, 16:22] = 0; m[18:20, 18:20] = 1; cases['island_inside_hole'] = m
m = np.zeros((64, 64), np.uint8); m[5, 5] = 1; cases['dot_1px'] = m
m = np.zeros((64, 64), np.uint8); m[30:32, 40:42] = 1; cases['dot_2px'] = m
for k, v in cases.items(): print(k, int(v.sum()), 'px')

### Step 1-2: components to polygons

Experiment: one component, contour on a 2x copy, coordinates back in original pixels (`(u + 0.5) / 2`, the centre of 2x pixel `u`). Verify by filling the polygon on the 2x grid and folding back.

In [ ]:
# experiment: contours on a 2x upsampled component
comp = (cases['small_dots'] == 1).astype(np.uint8)
n, lab = cv2.connectedComponents(comp, connectivity=8)
print('components:', n - 1)
up = cv2.resize((lab == 1).astype(np.uint8), None, fx=2, fy=2, interpolation=cv2.INTER_NEAREST)
cs, hier = cv2.findContours(up, cv2.RETR_CCOMP, cv2.CHAIN_APPROX_NONE)
poly = ((cs[0].reshape(-1, 2) + 0.5) / 2).reshape(-1)
print('polygon numbers:', len(poly), 'area (px^2):', round(cv2.contourArea(poly.reshape(-1, 1, 2).astype(np.float32)), 3))
# plain pixel-centre contour of the same 2x2 dot has area 1.0 only
plain, _ = cv2.findContours((lab == 1).astype(np.uint8), cv2.RETR_CCOMP, cv2.CHAIN_APPROX_NONE)
print('plain contour area:', cv2.contourArea(plain[0]))
assert cv2.contourArea(poly.reshape(-1, 1, 2).astype(np.float32)) > 0

### Step 3: RLE for holes

COCO RLE is column-major run lengths starting with a run of zeros. Experiment: encode, decode, compare.

In [ ]:
# experiment: uncompressed COCO RLE, column-major (Fortran order)
def _enc(m):
    flat = m.astype(np.uint8).T.reshape(-1)
    runs = np.diff(np.concatenate([[0], np.flatnonzero(np.diff(flat)) + 1, [flat.size]])).tolist()
    return {'counts': ([0] if flat[0] == 1 else []) + runs, 'size': [int(m.shape[0]), int(m.shape[1])]}
def _dec(r):
    h, w = r['size']; flat = np.zeros(h * w, np.uint8); pos = 0; val = 0
    for k in r['counts']:
        if val: flat[pos:pos + k] = 1
        pos += k; val ^= 1
    return flat.reshape(w, h).T
r = _enc(cases['square_with_hole'])
assert (_dec(r) == cases['square_with_hole']).all() and sum(r['counts']) == 64 * 64
print('rle runs:', len(r['counts']))

### Export: the encoders and the forward conversion

In [ ]:
#| export
def _rle_encode(
    binary_mask: np.ndarray,   # (H, W) {0,1}
) -> dict:
    "Uncompressed COCO RLE: column-major run lengths, first run counts zeros."
    column_major = (binary_mask > 0).astype(np.uint8).T.reshape(-1)
    run_starts = np.flatnonzero(np.diff(column_major)) + 1
    runs = np.diff(np.concatenate([[0], run_starts, [column_major.size]])).tolist()
    return {'counts': ([0] if column_major[0] == 1 else []) + runs,
            'size': [int(binary_mask.shape[0]), int(binary_mask.shape[1])]}

def _rle_decode(
    rle: dict,   # {'counts': list[int], 'size': [H, W]}
) -> np.ndarray:
    "Inverse of `_rle_encode`: (H, W) uint8 {0,1}."
    counts = rle['counts']
    if isinstance(counts, (str, bytes)):
        raise ValueError('compressed RLE strings are not supported; use the uncompressed counts list')
    height, width = rle['size']
    column_major = np.zeros(height * width, np.uint8)
    cursor = 0
    run_is_foreground = False
    for run in counts:
        if run_is_foreground:
            column_major[cursor:cursor + run] = 1
        cursor += run
        run_is_foreground = not run_is_foreground
    return column_major.reshape(width, height).T

def _component_polygons(
    comp: np.ndarray,    # (H, W) uint8 {0,1}, one connected component
    epsilon: float = 0,  # Douglas-Peucker tolerance in original pixels; 0 = lossless
) -> tuple:
    "Outer polygons (flat `[x1, y1, ...]` lists, original-pixel coords) and whether the component has holes."
    upscaled = cv2.resize(comp, None, fx=2, fy=2, interpolation=cv2.INTER_NEAREST)
    contours, hierarchy = cv2.findContours(upscaled, cv2.RETR_CCOMP, cv2.CHAIN_APPROX_NONE)
    has_holes = hierarchy is not None and bool((hierarchy[0][:, 3] >= 0).any())
    polys = []
    for contour, node in zip(contours, hierarchy[0]):
        parent = node[3]
        if parent >= 0:
            continue
        if epsilon > 0:
            contour = cv2.approxPolyDP(contour, epsilon * 2, True)
        polys.append(((contour.reshape(-1, 2).astype(np.float64) + 0.5) / 2).reshape(-1).tolist())
    return polys, has_holes

def mask_to_coco_instances(
    mask: np.ndarray,                  # (H, W) labels (0 = background, k = class k) or (K, H, W) multilabel {0,1}
    image_id: int = 1,                 # `image_id` written on every annotation
    first_ann_id: int = 1,             # id of the first annotation; the rest count up
    segmentation: str = 'auto',        # 'auto' polygon, RLE where a hole exists | 'polygon' (fills holes) | 'rle'
    epsilon: float = 0,                # polygon simplification in pixels; 0 keeps every contour point (lossless)
    min_area: int = 1,                 # drop components with fewer pixels
    connectivity: int = 8,             # 4 or 8
) -> list:
    """Split a mask into COCO instance annotations (one per connected component).

    `category_id` is the label value (or channel index + 1 for a `(K, H, W)` stack).
    `area` is the pixel count and `bbox` is `[x, y, w, h]`. Use `binarize` first for 0/255 masks.

    `segmentation='auto'` writes polygons, and an uncompressed-RLE dict only for components that have a hole
    (polygons cannot hold one). pycocotools reads that RLE (checked in the notebook); whether RF-DETR's own
    loader accepts it is unverified, so pass `segmentation='polygon'` (holes get filled) if it does not.

    Example:
        >>> m = np.zeros((8, 8), np.uint8); m[1:3, 1:3] = 1
        >>> a, = mask_to_coco_instances(m); a['category_id'], a['area'], a['bbox']
        (1, 4, [1, 1, 2, 2])
    """
    if segmentation not in ('auto', 'polygon', 'rle'):
        raise ValueError(f"segmentation must be 'auto', 'polygon' or 'rle', got {segmentation!r}")
    mask = np.asarray(mask)
    if mask.ndim == 3:
        layers = [(k + 1, mask[k] > 0) for k in range(mask.shape[0])]
    elif mask.ndim == 2:
        layers = [(int(v), mask == v) for v in np.unique(mask) if v > 0]
    else:
        raise ValueError(f'mask must be (H, W) or (K, H, W), got shape {mask.shape}')
    anns = []
    ann_id = first_ann_id
    for cat, layer in layers:
        n, lab = cv2.connectedComponents(layer.astype(np.uint8), connectivity=connectivity)
        for i in range(1, n):
            comp = (lab == i).astype(np.uint8)
            area = int(comp.sum())
            if area < min_area:
                continue
            ys, xs = np.nonzero(comp)
            bbox = [int(xs.min()), int(ys.min()), int(xs.max() - xs.min() + 1), int(ys.max() - ys.min() + 1)]
            seg = None
            if segmentation != 'rle':
                polys, has_holes = _component_polygons(comp, epsilon)
                if segmentation == 'polygon' or not has_holes:
                    seg = polys
            if seg is None:
                seg = _rle_encode(comp)
            anns.append({'id': ann_id, 'image_id': image_id, 'category_id': cat, 'segmentation': seg,
                         'area': area, 'bbox': bbox, 'iscrowd': 0})
            ann_id += 1
    return anns

In [ ]:
#| test
# test: shapes of the output, categories, RLE only where a hole exists, validation
a = mask_to_coco_instances(cases['small_dots'])
assert len(a) == 3 and [x['area'] for x in a] == [4, 4, 1] or sorted(x['area'] for x in a) == [1, 4, 4]
assert all(isinstance(x['segmentation'], list) and len(x['segmentation'][0]) >= 6 for x in a)
assert [x['id'] for x in a] == [1, 2, 3]
h, = mask_to_coco_instances(cases['square_with_hole'])
assert isinstance(h['segmentation'], dict) and h['area'] == 30 * 30 - 100 and h['bbox'] == [10, 10, 30, 30]
assert isinstance(mask_to_coco_instances(cases['square_with_hole'], segmentation='polygon')[0]['segmentation'], list)
assert isinstance(mask_to_coco_instances(cases['small_dots'], segmentation='rle')[0]['segmentation'], dict)
multi = np.zeros((16, 16), np.uint8); multi[1:4, 1:4] = 1; multi[8:12, 8:12] = 2
assert sorted(x['category_id'] for x in mask_to_coco_instances(multi)) == [1, 2]
stack = np.stack([multi == 1, multi == 2])
assert sorted(x['category_id'] for x in mask_to_coco_instances(stack)) == [1, 2]
assert mask_to_coco_instances(cases['small_dots'], min_area=2)[0]['area'] >= 2 and len(mask_to_coco_instances(cases['small_dots'], min_area=2)) == 2
assert mask_to_coco_instances(np.zeros((8, 8), np.uint8)) == []
try: mask_to_coco_instances(multi, segmentation='x'); raise AssertionError
except ValueError: pass

### Step 4: the inverse, and the round-trip measurement

`coco_instances_to_mask` rasterises polygons on the same 2x grid they were traced on (fill, then fold each 2x2 block back with `max`) and decodes RLE. Later annotations overwrite earlier ones where they overlap.

In [ ]:
#| export
def coco_instances_to_mask(
    anns: list,        # annotations from `mask_to_coco_instances` (or a COCO file)
    shape: tuple,      # (H, W) of the target mask
) -> np.ndarray:
    """Rasterise COCO instance annotations into an `(H, W)` int32 label map (`category_id` per pixel).

    Example:
        >>> m = np.zeros((8, 8), np.uint8); m[1:3, 1:3] = 1
        >>> coco_instances_to_mask(mask_to_coco_instances(m), (8, 8)).sum()
        4
    """
    h, w = shape
    out = np.zeros((h, w), np.int32)
    for a in anns:
        seg = a['segmentation']
        if isinstance(seg, dict):
            m = _rle_decode(seg).astype(bool)
        else:
            up = np.zeros((h * 2, w * 2), np.uint8)
            for poly in seg:
                pts = np.round(np.asarray(poly, np.float64).reshape(-1, 2) * 2 - 0.5).astype(np.int32)
                cv2.fillPoly(up, [pts], 1)
            m = up.reshape(h, 2, w, 2).max(axis=(1, 3)).astype(bool)
        out[m] = a['category_id']
    return out

In [ ]:
# experiment: measure the round trip per case (IoU of foreground, pixels that differ)
def _iou(a, b): return float((a & b).sum() / max((a | b).sum(), 1))
rows = {}
for name, m in cases.items():
    for mode in ('auto', 'polygon', 'rle'):
        back = coco_instances_to_mask(mask_to_coco_instances(m, segmentation=mode), m.shape) > 0
        rows[(name, mode)] = (_iou(back, m > 0), int((back != (m > 0)).sum()))
for (name, mode), (i, d) in rows.items(): print(f'{name:22s} {mode:8s} IoU={i:.4f}  wrong_px={d}')

**Measured loss per case** (this notebook, 64x64 synthetic masks, our own decoder `coco_instances_to_mask`; "wrong px" counts pixels that differ from the original):

| case | `auto` (default) | `polygon` | `rle` |
|---|---|---|---|
| 1-px thin diagonal line | IoU 1.0000, 0 wrong px | 1.0000, 0 | 1.0000, 0 |
| 1-px thin horizontal line | 1.0000, 0 | 1.0000, 0 | 1.0000, 0 |
| 1-px dot | 1.0000, 0 | 1.0000, 0 | 1.0000, 0 |
| 2-px dot (2x2) | 1.0000, 0 | 1.0000, 0 | 1.0000, 0 |
| square with a hole | 1.0000, 0 (uses RLE) | **0.8889, 100 wrong px**: the hole is filled (polygons cannot hold holes) | 1.0000, 0 |
| island inside a hole | 1.0000, 0 (RLE for the ring, polygon for the island) | **0.9339, 32 wrong px**: the ring's 36-px hole is filled, the 4-px island survives | 1.0000, 0 |

With `epsilon=0` (default) the polygon path is lossless on the pixel grid, so the default mode loses nothing on these cases. The lossy options are `segmentation='polygon'` (fills holes) and `epsilon > 0`. Measured on a radius-20 disc (552 polygon numbers at `epsilon=0`): `epsilon=0.5` gives 104 numbers and IoU 0.975, `epsilon=1` gives 32 numbers and IoU 0.950, `epsilon=2` gives 16 numbers and IoU 0.944. Do not simplify masks that carry 1-3 px defects.

**Real-consumer check (recorded, not a test):** the same annotations were rasterised with `pycocotools` (`mask.frPyObjects` + `mask.merge` + `mask.decode`, the path COCO loaders use) in a throwaway venv, since `pycocotools` is not a dependency of this repo. We write polygon vertices at half-pixel offsets, so this could have differed from our decoder; it does not:

| case | `auto` (default) | `polygon` | `rle` |
|---|---|---|---|
| 1-px thin diagonal line | IoU 1.0000, 0 wrong px | 1.0000, 0 | 1.0000, 0 |
| 1-px thin horizontal line | 1.0000, 0 | 1.0000, 0 | 1.0000, 0 |
| 1-px dot | 1.0000, 0 | 1.0000, 0 | 1.0000, 0 |
| 2-px dot (2x2) | 1.0000, 0 | 1.0000, 0 | 1.0000, 0 |
| square with a hole | 1.0000, 0 | 0.8889, 100 | 1.0000, 0 |
| island inside a hole | 1.0000, 0 | 0.9339, 32 | 1.0000, 0 |

Identical to the table above, so the polygon/offset convention needed no change. Two things this still does not cover: pycocotools reads the uncompressed RLE counts we write (the `auto` and `rle` columns), but **RF-DETR's own loader on RLE is unverified**, and the loader's resize to training resolution (312-768 px) is not modelled.

In [ ]:
#| test
# test: acceptance (issue #101): round-trip IoU >= 0.99 for thin lines, holes and 2-px dots in default mode,
# for every mode that can hold the feature; polygon mode is documented as lossy on holes only.
for name, m in cases.items():
    back = coco_instances_to_mask(mask_to_coco_instances(m), m.shape) > 0
    assert _iou(back, m > 0) >= 0.99, (name, _iou(back, m > 0))
    assert (back == (m > 0)).all(), name   # exact with epsilon=0
    assert _iou(coco_instances_to_mask(mask_to_coco_instances(m, segmentation='rle'), m.shape) > 0, m > 0) == 1.0
for name in ('thin_diagonal_line', 'thin_horizontal_line', 'small_dots'):
    m = cases[name]
    assert _iou(coco_instances_to_mask(mask_to_coco_instances(m, segmentation='polygon'), m.shape) > 0, m > 0) >= 0.99
hole_poly = coco_instances_to_mask(mask_to_coco_instances(cases['square_with_hole'], segmentation='polygon'), (64, 64)) > 0
assert 0.88 < _iou(hole_poly, cases['square_with_hole'] > 0) < 0.89   # holes filled: the documented loss
# multiclass round trip keeps the labels
multi = np.zeros((32, 32), np.uint8); multi[2:10, 2:10] = 1; multi[15:30, 15:30] = 2; multi[20:24, 20:24] = 0
assert (coco_instances_to_mask(mask_to_coco_instances(multi), multi.shape) == multi).all()
# simplification: smaller JSON, measured loss on a round blob
blob = np.zeros((64, 64), np.uint8); cv2.circle(blob, (32, 32), 20, 1, -1)
full = mask_to_coco_instances(blob); simp = mask_to_coco_instances(blob, epsilon=0.5)
n_full, n_simp = len(full[0]['segmentation'][0]), len(simp[0]['segmentation'][0])
iou_simp = _iou(coco_instances_to_mask(simp, blob.shape) > 0, blob > 0)
print(f'epsilon=0.5 on a r=20 disc: {n_full} -> {n_simp} numbers, IoU {iou_simp:.4f}')
assert n_simp < n_full and iou_simp >= 0.97

### Step 5: a COCO folder from the patch cache

The patch cache (`build_patch_dataset`) stores `patch_dir/{train,val}/images/{id}.png` and `masks/{id}.png`. RF-DETR wants `train/` and `valid/`, each with its images and one `_annotations.coco.json` beside them. Experiment on a tiny synthetic cache.

In [ ]:
# experiment: a synthetic patch cache in a temp dir, then the writer's job step by step
import json, tempfile
from pathlib import Path
tmp = Path(tempfile.mkdtemp())
for split, n in (('train', 3), ('val', 2)):
    (tmp / 'cache' / split / 'images').mkdir(parents=True); (tmp / 'cache' / split / 'masks').mkdir(parents=True)
    for i in range(n):
        img = np.full((48, 48), 40 + i, np.uint8); mk = np.zeros((48, 48), np.uint8)
        if i != 1: mk[10 + i:14 + i, 5:40] = 255; mk[30:32, 30:32] = 255
        cv2.imwrite(str(tmp / 'cache' / split / 'images' / f'p{i}.png'), img)
        cv2.imwrite(str(tmp / 'cache' / split / 'masks' / f'p{i}.png'), mk)
print(sorted(p.name for p in (tmp / 'cache' / 'train' / 'masks').glob('*.png')))

In [ ]:
#| export
def _to_uint8(img: np.ndarray) -> np.ndarray:
    "8-bit view of a patch image: 16-bit is divided by 257, anything else is clipped to 0-255."
    if img.dtype == np.uint8:
        return img
    if img.dtype == np.uint16:
        return (img / 257).round().astype(np.uint8)
    return np.clip(img, 0, 255).astype(np.uint8)

def write_coco_folder(
    patch_dir: str,                 # patch cache from `build_patch_dataset`: `{split}/images/{id}.png`, `{split}/masks/{id}.png`
    out_dir: str,                   # where to write `train/`, `valid/` with `_annotations.coco.json`
    class_names: Sequence[str],     # foreground names; category ids are 1..K (one name = binary mask)
    splits: Optional[dict] = None,  # {cache split: output folder}, default `{'train': 'train', 'val': 'valid'}`
    **kw,                           # forwarded to `mask_to_coco_instances` (segmentation, epsilon, min_area, ...)
) -> dict:
    """Write the patch cache as a COCO instance dataset (8-bit PNGs + `_annotations.coco.json`) and return `{folder: (n_images, n_annotations)}`.

    Empty patches are kept (with no annotations). Single-channel caches only: a `_ch0`-style multi-channel cache raises.
    With K `class_names`, mask values must lie in 0..K (a larger value raises `ValueError`); one name binarizes the mask.
    16-bit images are divided by 257: this is **lossy** (65536 grey levels become 256), so contrast that lives in the low
    bits is gone from the COCO copy. `segmentation='auto'` writes RLE only for components with holes; RF-DETR's loader
    on that RLE is unverified (see `mask_to_coco_instances`).

    Example:
        >>> write_coco_folder('patches', 'coco', ['defect'])  # doctest: +SKIP
        {'train': (120, 310), 'valid': (30, 77)}
    """
    splits = splits or {'train': 'train', 'val': 'valid'}
    names = list(class_names)
    summary = {}
    for src, dst in splits.items():
        mdir, idir = Path(patch_dir) / src / 'masks', Path(patch_dir) / src / 'images'
        out = Path(out_dir) / dst
        out.mkdir(parents=True, exist_ok=True)
        images, annotations, ann_id = [], [], 1
        for image_id, mpath in enumerate(sorted(mdir.glob('*.png')), start=1):
            ipath = idir / mpath.name
            if not ipath.exists():
                raise FileNotFoundError(f'{ipath} missing: multi-channel caches (`_ch0` images) are not supported')
            img = cv2.imread(str(ipath), cv2.IMREAD_UNCHANGED)
            mask = cv2.imread(str(mpath), cv2.IMREAD_UNCHANGED)
            if mask.ndim == 3:
                raise ValueError(f'{mpath} has {mask.shape[2]} channels; stacked masks are not supported')
            if len(names) == 1:
                mask = binarize(mask)
            elif int(mask.max()) > len(names):
                raise ValueError(f'{mpath} has mask value {int(mask.max())} but only {len(names)} class_names '
                                 f'{names}; category ids would have no entry in `categories`')
            cv2.imwrite(str(out / mpath.name), _to_uint8(img))
            images.append({'id': image_id, 'file_name': mpath.name, 'width': int(img.shape[1]), 'height': int(img.shape[0])})
            anns = mask_to_coco_instances(mask, image_id=image_id, first_ann_id=ann_id, **kw)
            ann_id += len(anns)
            annotations.extend(anns)
        cats = [{'id': k + 1, 'name': n} for k, n in enumerate(names)]
        (out / '_annotations.coco.json').write_text(json.dumps(
            {'images': images, 'annotations': annotations, 'categories': cats}))
        summary[dst] = (len(images), len(annotations))
    return summary

In [ ]:
#| test
# test: the writer on the synthetic cache: folders, file names, counts, and a lossless mask round trip
res = write_coco_folder(str(tmp / 'cache'), str(tmp / 'coco'), ['defect'])
assert res == {'train': (3, 4), 'valid': (2, 2)}, res
coco = json.loads((tmp / 'coco' / 'train' / '_annotations.coco.json').read_text())
assert [c['name'] for c in coco['categories']] == ['defect'] and coco['categories'][0]['id'] == 1
assert {i['file_name'] for i in coco['images']} == {'p0.png', 'p1.png', 'p2.png'}
assert len({a['id'] for a in coco['annotations']}) == len(coco['annotations'])
for im in coco['images']:
    pic = cv2.imread(str(tmp / 'coco' / 'train' / im['file_name']), cv2.IMREAD_UNCHANGED)
    assert pic.dtype == np.uint8 and pic.shape == (im['height'], im['width'])
    gt = binarize(cv2.imread(str(tmp / 'cache' / 'train' / 'masks' / im['file_name']), cv2.IMREAD_UNCHANGED))
    anns = [a for a in coco['annotations'] if a['image_id'] == im['id']]
    assert (coco_instances_to_mask(anns, gt.shape) == gt).all()
# 16-bit images become 8-bit, a multi-channel cache is refused
assert _to_uint8(np.full((2, 2), 257 * 100, np.uint16)).max() == 100
import shutil
bad = tmp / 'bad'; shutil.copytree(tmp / 'cache', bad); (bad / 'train' / 'images' / 'p0.png').rename(bad / 'train' / 'images' / 'p0_ch0.png')
try: write_coco_folder(str(bad), str(tmp / 'o'), ['defect']); raise AssertionError
except FileNotFoundError: pass

In [ ]:
#| test
# test: multi-class folder layout, categories 1..K, empty patches kept, mask value > K and stacked masks refused
mc = tmp / 'mc'
for split, vals in (('train', (0, 1, 2)), ('val', (0, 1))):
    (mc / split / 'images').mkdir(parents=True); (mc / split / 'masks').mkdir(parents=True)
    for j, v in enumerate(vals):
        cv2.imwrite(str(mc / split / 'images' / f'q{j}.png'), np.full((32, 32), 9, np.uint16 if j == 2 else np.uint8))
        mk = np.zeros((32, 32), np.uint8); mk[4:9, 4:20] = v
        cv2.imwrite(str(mc / split / 'masks' / f'q{j}.png'), mk)
res = write_coco_folder(str(mc), str(tmp / 'mc_out'), ['chip', 'void'])
assert res == {'train': (3, 2), 'valid': (2, 1)}, res
assert sorted(p.name for p in (tmp / 'mc_out').iterdir()) == ['train', 'valid']
for folder in ('train', 'valid'):
    assert (tmp / 'mc_out' / folder / '_annotations.coco.json').is_file()
    assert all(p.suffix in ('.png', '.json') for p in (tmp / 'mc_out' / folder).iterdir())
coco = json.loads((tmp / 'mc_out' / 'train' / '_annotations.coco.json').read_text())
assert coco['categories'] == [{'id': 1, 'name': 'chip'}, {'id': 2, 'name': 'void'}]
assert sorted(a['category_id'] for a in coco['annotations']) == [1, 2]
assert len(coco['images']) == 3 and 'q0.png' in {i['file_name'] for i in coco['images']}   # empty patch q0 kept
assert {a['image_id'] for a in coco['annotations']} == {2, 3}
assert cv2.imread(str(tmp / 'mc_out' / 'train' / 'q2.png'), cv2.IMREAD_UNCHANGED).dtype == np.uint8
# one class name means a binary mask (0/1 or 0/255): `binarize` is applied, no K check
assert write_coco_folder(str(tmp / 'cache'), str(tmp / 'one'), ['defect']) == {'train': (3, 4), 'valid': (2, 2)}
# a mask value above K would give a category_id with no entry in `categories`
bad_k = tmp / 'bad_k'; shutil.copytree(mc, bad_k)
cv2.imwrite(str(bad_k / 'train' / 'masks' / 'q2.png'), np.full((32, 32), 3, np.uint8))
try: write_coco_folder(str(bad_k), str(tmp / 'o2'), ['chip', 'void']); raise AssertionError('value 3 with K=2 must raise')
except ValueError as e: assert '3' in str(e) and '2' in str(e), e
# stacked (3-channel) mask refused
st_ = tmp / 'st'; shutil.copytree(mc, st_)
cv2.imwrite(str(st_ / 'train' / 'masks' / 'q1.png'), np.zeros((32, 32, 3), np.uint8))
try: write_coco_folder(str(st_), str(tmp / 'o3'), ['chip', 'void']); raise AssertionError
except ValueError as e: assert 'channels' in str(e)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()